In [44]:
#### BL: baseline, NTC: no target siRNA control. Drop-seq: 另外一种scRNA测序技术。
#### KD: knockdown
#### geo中有scenic+处理好的pseudo-bulk数据.

In [8]:
import pandas as pd
import anndata as ad
import numpy as np
from scipy.sparse import csr_matrix
import re

In [3]:
scenic_mm_gene = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/melanoma/scATAC/GSE210745_mm_lines_gene_expression_matrix.tsv.gz",sep='\t',header=0,index_col=0)

In [4]:
scenic_mm_atac = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/melanoma/scATAC/GSE210745_mm_lines_region_accessibility_matrix.tsv.gz",sep='\t',header=0,index_col=0)

In [5]:
scenic_mm_gene = (scenic_mm_gene*5).astype(int)
scenic_mm_atac = (scenic_mm_atac*5).astype(int)

In [ ]:
peaks = [re.split(r'[:\-]',i) for i in scenic_mm_atac.index.values]
peaks2 = [f"{i[0]}-{i[1]}-{i[2]}" for i in peaks]


['chr6-146944773-146945273',
 'chr15-48634866-48635366',
 'chr19-16814090-16814590',
 'chr2-36057979-36058479',
 'chr6-18551190-18551690',
 'chr4-175412096-175412596',
 'chr4-173835924-173836424',
 'chr6-36933060-36933560',
 'chrX-45557463-45557963',
 'chr15-40498303-40498803',
 'chr2-190115529-190116029',
 'chr2-11354217-11354717',
 'chr5-159449877-159450377',
 'chr2-118386896-118387396',
 'chr10-119738276-119738776',
 'chr11-62679828-62680328',
 'chr10-34295112-34295612',
 'chrX-15739759-15740259',
 'chr3-15398813-15399313',
 'chr5-89396866-89397366',
 'chr10-107546903-107547403',
 'chr8-104355674-104356174',
 'chr9-127714871-127715371',
 'chr19-12628024-12628524',
 'chr12-68416733-68417233',
 'chr21-26055831-26056331',
 'chr2-151849514-151850014',
 'chr4-95739599-95740099',
 'chr17-64788103-64788603',
 'chr3-127679299-127679799',
 'chr21-43697692-43698192',
 'chr20-60295208-60295708',
 'chr7-141356199-141356699',
 'chr12-23535381-23535881',
 'chr20-48552171-48552671',
 'chr14-930927

In [12]:
scenic_mm_meta = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/melanoma/scATAC/GSE210745_mm_lines_cell_metadata.tsv",sep='\t',header=0,index_col=0)

In [13]:
scenic_mm_meta['lineState'].value_counts()

lineState
INT    358
MEL    318
MES    260
Name: count, dtype: int64

In [14]:
metadata = pd.DataFrame({"barcode":scenic_mm_meta.index.values,"batch":scenic_mm_meta['MMline'],"cell_type":scenic_mm_meta["lineState"]})
metadata.to_csv("/home/shaliu_fu/multireg/benchmark/bench_dataset/melanoma/metadata.csv",sep=",",header=True,index=False)

In [13]:
rna_h5ad = ad.AnnData(X=csr_matrix(scenic_mm_gene.values),obs=metadata,var=pd.DataFrame(index=scenic_mm_gene.columns.values))
rna_h5ad.write_h5ad("/home/shaliu_fu/multireg/benchmark/bench_dataset/melanoma/Melanoma-cell_line-RNA-counts.h5ad")

In [15]:
rna_h5ad.X

<936x17085 sparse matrix of type '<class 'numpy.int64'>'
	with 5694302 stored elements in Compressed Sparse Row format>

In [15]:
# atac_h5ad = ad.AnnData(X=csr_matrix(scenic_mm_atac.T.values), obs=metadata, var=pd.DataFrame(index=scenic_mm_atac.index.values))
atac_h5ad = ad.AnnData(X=csr_matrix(scenic_mm_atac.T.values), obs=metadata, var=pd.DataFrame(index=peaks2))
atac_h5ad.write_h5ad("/home/shaliu_fu/multireg/benchmark/bench_dataset/melanoma/Melanoma-cell_line-ATAC-peaks.h5ad")

In [42]:
import json

out_json={
  "rna_h5ad_filename": "Melanoma-cell_line-RNA-counts.h5ad", 
  "atac_h5ad_filename": "Melanoma-cell_line-ATAC-peaks.h5ad",
  "rna_rds_filename": "Melanoma-cell_line-RNA-counts.rds",
  "atac_rds_filename": "Melanoma-cell_line-ATAC-peaks.rds",
  "metadata": "metadata.csv",
  "barcode_key": "barcode",
  "celltype_key": "cell_type",
  "batch_key": "batch",
  "output_prefix": "Melanoma-cell_line-scRNA+scATAC",
  "species": "human",
  "gene_id": "symbol",
  "LSI_file":"Melanoma-cell_line-lsi.txt",
  "gtf_file": "/home/shaliu_fu/multireg/benchmark/db/gencode.v41.chr_patch_hapl_scaff.annotation.gtf.gz",
}
with open('/home/shaliu_fu/multireg/benchmark/bench_dataset/melanoma/RawData.json', 'w', encoding='utf-8') as f:
    json.dump(out_json, f, ensure_ascii=False, indent=4)